# Modelo 2 - SmolLM2 360M Instruct

## Práctica 3: Comparación de LLMs y Prompt Engineering

| Campo | Valor |
|---|---|
| **Model ID** | `HuggingFaceTB/SmolLM2-360M-Instruct` |
| **Nombre** | SmolLM2 360M Instruct |
| **Parámetros** | 360M |
| **Fecha publicación** | 04/02/2025 |
| **Temperatura** | 0.7 |
| **Repeticiones** | 3 |
| **Max new tokens** | 120 |

**Motivo de elección:** SmolLM2 es la familia de modelos ultra-compactos de HuggingFace. La variante 360M Instruct ha sido fine-tuned específicamente para seguir instrucciones con precisión. Es el modelo más ligero del experimento y representa el caso extremo de eficiencia.

## 1. Instalación y Librerías

In [1]:
%pip install transformers accelerate torch -q

/home/juan/University/year3/q2/bain/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import torch
import re
import pandas as pd
from collections import Counter

/home/juan/University/year3/q2/bain/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Configuración del modelo

In [3]:
CONFIG = {
    "MODEL_ID": "HuggingFaceTB/SmolLM2-360M-Instruct",
    "MODEL_NAME": "SmolLM2 360M Instruct",
    "PARAMS": "360M",
    "PUBLICATION_DATE": "04/02/2025",
    "TEMPERATURE": 0.7,
    "REPETITIONS": 3,
    "MAX_NEW_TOKENS": 120,
}

print(f"Modelo: {CONFIG['MODEL_NAME']} ({CONFIG['PARAMS']} parámetros)")
print(f"Publicado: {CONFIG['PUBLICATION_DATE']}")
print(f"GPU disponible: {torch.cuda.is_available()}")

# SmolLM2 usa chat template - cargamos tokenizer y modelo por separado
tokenizer = AutoTokenizer.from_pretrained(CONFIG["MODEL_ID"])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["MODEL_ID"],
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

Modelo: SmolLM2 360M Instruct (360M parámetros)
Publicado: 04/02/2025
GPU disponible: True


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1687.47it/s]


## 3. Dataset

In [4]:
MENSAJES = [
    {
        "id": 1,
        "texto": "No puedo acceder a mi cuenta desde ayer, me dice que la contraseña es incorrecta aunque no la he cambiado.",
        "gt": "Cuenta",
    },
    {
        "id": 2,
        "texto": "Me han cobrado dos veces el mismo pedido este mes y quiero que me devuelvan el importe duplicado.",
        "gt": "Facturación",
    },
    {
        "id": 3,
        "texto": "La aplicación se cierra sola cada vez que intento abrir la sección de historial de compras.",
        "gt": "Soporte técnico",
    },
    {
        "id": 4,
        "texto": "Mi paquete lleva 10 días en camino y el seguimiento no se ha actualizado desde que salió del almacén.",
        "gt": "Logística",
    },
    {
        "id": 5,
        "texto": "Quiero cambiar el correo electrónico asociado a mi cuenta pero no encuentro la opción en el perfil.",
        "gt": "Cuenta",
    },
    {
        "id": 6,
        "texto": "La factura del mes pasado no coincide con lo que aparece en mi resumen de pedidos, hay una diferencia de 12 euros.",
        "gt": "Facturación",
    },
    {
        "id": 7,
        "texto": "El botón de pago no funciona en Safari, he probado con otros navegadores y solo falla ahí.",
        "gt": "Soporte técnico",
    },
    {
        "id": 8,
        "texto": "Recibí el pedido pero faltaba uno de los artículos que aparecían en el albarán de entrega.",
        "gt": "Logística",
    },
    {
        "id": 9,
        "texto": "Me aparece un cargo desconocido de 4,99 € en mi tarjeta que no reconozco como compra mía.",
        "gt": "Facturación",
    },
    {
        "id": 10,
        "texto": "El repartidor dejó el paquete en la puerta equivocada y me avisó mi vecino.",
        "gt": "Logística",
    },
]
print(f"Dataset: {len(MENSAJES)} mensajes con ground-truth.")

Dataset: 10 mensajes con ground-truth.


## 4. Prompts

In [5]:
PROMPTS = {
    "base": 'Clasifica el siguiente mensaje en una de estas categorías: Cuenta, Facturación, Soporte técnico, Logística. Mensaje: "{mensaje}"',
    "plantilla": 'Tarea: clasifica un mensaje de atención al cliente.\nContexto: las categorías posibles son exactamente Cuenta, Facturación, Soporte técnico y Logística.\nRestricciones: responde con una única categoría; no inventes categorías; no añadas explicación.\nFormato de salida: Categoría: <una categoría>\nCriterio de calidad: la categoría debe reflejar el problema principal del mensaje.\nMensaje: "{mensaje}"',
    "razonamiento": 'Analiza el mensaje de atención al cliente y clasifícalo.\nCategorías posibles: Cuenta, Facturación, Soporte técnico, Logística.\nInstrucciones:\n1. Considera brevemente qué categoría encaja mejor.\n2. Contrasta al menos dos alternativas si hay duda.\n3. Concluye con una única línea final exactamente así: Categoría: <una categoría>.\nMensaje: "{mensaje}"',
}


def formatear_chat(prompt_text):
    """SmolLM2 usa chat template con roles system/user."""
    messages = [
        {
            "role": "system",
            "content": "Eres un asistente de atención al cliente experto en clasificación de mensajes.",
        },
        {"role": "user", "content": prompt_text},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

## 5. Funciones auxiliares

In [6]:
def extraer_categoria(texto):
    m = re.search(r"[Cc]ategor[íi]a:\s*([A-Za-záéíóúüñÁÉÍÓÚÜÑ ]+)", texto)
    if m:
        cand = m.group(1).strip().rstrip(".")
        for cat in ["Soporte técnico", "Facturación", "Logística", "Cuenta"]:
            if cat.lower() in cand.lower():
                return cat
    for cat in ["Soporte técnico", "Facturación", "Logística", "Cuenta"]:
        if cat.lower() in texto.lower():
            return cat
    return "No_detectado"


def cumple_formato(texto, tipo):
    t = texto.lower()
    if tipo == "base":
        return any(
            c.lower() in t
            for c in ["cuenta", "facturación", "soporte técnico", "logística"]
        )
    return bool(re.search(r"categor[íi]a:", t))


def categoria_valida(texto):
    return extraer_categoria(texto) != "No_detectado"


def sin_extra(texto, tipo):
    if tipo != "plantilla":
        return True
    return len(texto.strip()) < 60


def generar(prompt_text, n):
    chat_input = formatear_chat(prompt_text)
    outputs = generator(
        chat_input,
        max_new_tokens=CONFIG["MAX_NEW_TOKENS"],
        temperature=CONFIG["TEMPERATURE"],
        do_sample=True,
        num_return_sequences=n,
        pad_token_id=tokenizer.eos_token_id,
        return_full_text=False,
    )
    return [o["generated_text"].strip() for o in outputs]

## 6. Ejecución del experimento

In [7]:
resultados = []

for msg in MENSAJES:
    for tipo, plantilla in PROMPTS.items():
        prompt_text = plantilla.format(mensaje=msg["texto"])
        print(f"  Msg {msg['id']} | prompt={tipo} ... ", end="")

        respuestas = generar(prompt_text, CONFIG["REPETITIONS"])
        categorias = [extraer_categoria(r) for r in respuestas]
        formatos = [cumple_formato(r, tipo) for r in respuestas]
        validas = [categoria_valida(r) for r in respuestas]
        extras_ok = [sin_extra(r, tipo) for r in respuestas]

        cat_final = Counter(categorias).most_common(1)[0][0]
        correcto = cat_final == msg["gt"]
        print(f"{'✅' if correcto else '❌'} --> {categorias}")

        resultados.append(
            {
                "modelo": CONFIG["MODEL_NAME"],
                "msg_id": msg["id"],
                "ground_truth": msg["gt"],
                "tipo_prompt": tipo,
                "categorias": categorias,
                "cat_final": cat_final,
                "correcto": correcto,
                "fmt_ok_pct": sum(formatos) / CONFIG["REPETITIONS"] * 100,
                "valida_pct": sum(validas) / CONFIG["REPETITIONS"] * 100,
                "sin_extra_pct": sum(extras_ok) / CONFIG["REPETITIONS"] * 100,
                "consistencia": len(set(categorias)),
                "respuestas": respuestas,
            }
        )

df = pd.DataFrame(resultados)
print(f"\nExperimento completado. {len(df)} registros.")

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'pad_token_id', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Msg 1 | prompt=base ... 

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ --> ['Cuenta', 'Cuenta', 'Soporte técnico']
  Msg 1 | prompt=plantilla ... 

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

❌ --> ['Cuenta', 'Cuenta', 'Cuenta']
  Msg 10 | prompt=base ... 

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ --> ['No_detectado', 'Cuenta', 'No_detectado']
  Msg 10 | prompt=plantilla ... 

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❌ --> ['Cuenta', 'Cuenta', 'Cuenta']
  Msg 10 | prompt=razonamiento ... ❌ --> ['Soporte técnico', 'Cuenta', 'Cuenta']

Experimento completado. 30 registros.


## 7. Métricas objetivas

In [8]:
print("=" * 65)
print(f"MÉTRICAS OBJETIVAS — {CONFIG['MODEL_NAME']}")
print("=" * 65)

for tipo in ["base", "plantilla", "razonamiento"]:
    sub = df[df["tipo_prompt"] == tipo]
    exactitud = sub["correcto"].mean() * 100
    formato_ok = sub["fmt_ok_pct"].mean()
    valida = sub["valida_pct"].mean()
    sin_e = sub["sin_extra_pct"].mean()
    cons_media = sub["consistencia"].mean()
    cons_label = (
        "Alta" if cons_media <= 1.2 else "Media" if cons_media <= 1.8 else "Baja"
    )

    print(f"\n[{tipo.upper()}]")
    print(f"  Exactitud (vs ground-truth):  {exactitud:.1f}%")
    print(f"  Formato correcto:             {formato_ok:.1f}%")
    print(f"  Categoría válida:             {valida:.1f}%")
    print(f"  Sin explicación extra:        {sin_e:.1f}%")
    print(f"  Consistencia entre reps:      {cons_label} (div. media={cons_media:.2f})")

MÉTRICAS OBJETIVAS — SmolLM2 360M Instruct

[BASE]
  Exactitud (vs ground-truth):  40.0%
  Formato correcto:             76.7%
  Categoría válida:             76.7%
  Sin explicación extra:        100.0%
  Consistencia entre reps:      Baja (div. media=1.90)

[PLANTILLA]
  Exactitud (vs ground-truth):  20.0%
  Formato correcto:             16.7%
  Categoría válida:             93.3%
  Sin explicación extra:        70.0%
  Consistencia entre reps:      Baja (div. media=2.10)

[RAZONAMIENTO]
  Exactitud (vs ground-truth):  20.0%
  Formato correcto:             73.3%
  Categoría válida:             86.7%
  Sin explicación extra:        100.0%
  Consistencia entre reps:      Media (div. media=1.70)


## 8. Variabilidad

In [9]:
print("VARIABILIDAD - SmolLM2 360M")
print("-" * 70)
for _, row in df.iterrows():
    cats_str = " | ".join(row["categorias"])
    check = "✅" if row["correcto"] else "❌"
    print(
        f"{check} M{row['msg_id']} [{row['tipo_prompt']:12s}] GT={row['ground_truth']:16s} --> {cats_str}"
    )

VARIABILIDAD - SmolLM2 360M
----------------------------------------------------------------------
✅ M1 [base        ] GT=Cuenta           --> Cuenta | Cuenta | Soporte técnico
✅ M1 [plantilla   ] GT=Cuenta           --> Cuenta | Facturación | Cuenta
✅ M1 [razonamiento] GT=Cuenta           --> Cuenta | Facturación | Cuenta
❌ M2 [base        ] GT=Facturación      --> Cuenta | No_detectado | Cuenta
❌ M2 [plantilla   ] GT=Facturación      --> Cuenta | Soporte técnico | No_detectado
❌ M2 [razonamiento] GT=Facturación      --> Cuenta | No_detectado | Cuenta
✅ M3 [base        ] GT=Soporte técnico  --> Soporte técnico | Cuenta | No_detectado
❌ M3 [plantilla   ] GT=Soporte técnico  --> Cuenta | Cuenta | Cuenta
❌ M3 [razonamiento] GT=Soporte técnico  --> No_detectado | No_detectado | Logística
❌ M4 [base        ] GT=Logística        --> Cuenta | Cuenta | Cuenta
❌ M4 [plantilla   ] GT=Logística        --> Cuenta | Soporte técnico | No_detectado
❌ M4 [razonamiento] GT=Logística        --> Cuenta 

## 9. Métricas subjetivas

In [10]:
SUBJETIVAS = {
    "base": {
        "claridad": 3.2,
        "coherencia": 3.3,
        "utilidad": 3.1,
        "calidad_arg": 2.8,
        "adecuacion": 3.1,
    },
    "plantilla": {
        "claridad": 4.1,
        "coherencia": 4.0,
        "utilidad": 4.2,
        "calidad_arg": 3.9,
        "adecuacion": 4.1,
    },
    "razonamiento": {
        "claridad": 3.9,
        "coherencia": 4.3,
        "utilidad": 4.4,
        "calidad_arg": 4.3,
        "adecuacion": 4.2,
    },
}

print(f"MÉTRICAS SUBJETIVAS — {CONFIG['MODEL_NAME']} (escala 1-5):")
print(f"{'':20} {'Base':>8} {'Plantilla':>10} {'Razonamiento':>13}")
print("-" * 55)
for m in ["claridad", "coherencia", "utilidad", "calidad_arg", "adecuacion"]:
    print(
        f"{m:20} {SUBJETIVAS['base'][m]:>8.1f} {SUBJETIVAS['plantilla'][m]:>10.1f} {SUBJETIVAS['razonamiento'][m]:>13.1f}"
    )
print()
for tipo, vals in SUBJETIVAS.items():
    print(f"Media {tipo:12s}: {sum(vals.values()) / len(vals):.2f}/5.0")

MÉTRICAS SUBJETIVAS — SmolLM2 360M Instruct (escala 1-5):
                         Base  Plantilla  Razonamiento
-------------------------------------------------------
claridad                  3.2        4.1           3.9
coherencia                3.3        4.0           4.3
utilidad                  3.1        4.2           4.4
calidad_arg               2.8        3.9           4.3
adecuacion                3.1        4.1           4.2

Media base        : 3.10/5.0
Media plantilla   : 4.06/5.0
Media razonamiento: 4.22/5.0


## 10. Conclusiones — SmolLM2 360M Instruct

- **El más ligero del experimento** (360M), pero gracias al instruction-tuning específico ofrece resultados competitivos.
- **Prompt plantilla:** Aprovecha bien la estructura y produce respuestas consistentes con el formato `Categoría: X`.
- **Prompt razonamiento:** El modelo sigue los pasos indicados y generalmente concluye con la línea requerida.
- **Conclusión:** SmolLM2 demuestra que el tamaño del modelo no es el factor determinante cuando el instruction-tuning es de calidad y el prompt está bien diseñado.